# 1. Spatially-informed bivariate metrics

This tutorial provides an overview of the local and global scores implemented in LIANA+. These scores are used to identify spatially co-expressed ligand-receptor pairs. However, they are also applicable to other types of spatially-informed bivariate analyses.
It briefly explains how local and global scores work, using weighted cosine similarity and bivariate Moran’s R as examples. The tutorial also showcases interaction categories (masks) and significance testing.

### Environment Setup

In [ ]:
import pandas as pd
import scanpy as sc
import squidpy as sq
import decoupler as dc
import liana as li
from matplotlib import pyplot as plt
import warnings
# silence squidpy's FutureWarnings, which otherwise bury the plots
warnings.filterwarnings('ignore', category=FutureWarning)
# set dpi to 100, to make the notebook smaller
plt.rcParams['figure.dpi'] = 100

### Load and Normalize Data

In [ ]:
import os
# li.ds caches downloads under ../../../datasets/Hands_on_2_LIANA_MistY/anndata; a provided Visium_19_CK297.h5ad there is used as-is
sc.settings.datasetdir = os.path.join(os.environ.get("MOPITAS_DATA", "../../../datasets"), "Hands_on_2_LIANA_MistY")
adata = li.ds.kuppe_2022()
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

### Available spatial correlation/similarity Functions

In [ ]:
li.mt.bivariate.show_functions()

### How do the local functions work?

The local functions work are quite simple, as they are simply weighted versions of well-known similarity metrics. For example, the spatially-weighted version of Cosine similarity is defined as:

$$ \text{wCosine}_i = \frac{\sum_{j=1}^n w_{ij} x_j y_j}{\sqrt{\sum_{j=1}^n w_{ij} x_j^2} \sqrt{\sum_{j=1}^n w_{ij} y_j^2}}$$

where for each spot **i**, we perform summation over all spots **n**, where **w**​ represents the spatial connectivity weights from spot **i** to every other spot **j**;  for variables  **x** and **y**.

### How do the global functions work?

Global scores can be calculated as the average local score across all spots for each ligand-receptor pair.

In addition, we can also use Global bivariate Moran's R (or [Lee's statistic](https://onlinelibrary.wiley.com/doi/full/10.1111/gean.12106)) - an extension of univariate Moran's I, as proposed by [Anselin 2019](https://onlinelibrary.wiley.com/doi/full/10.1111/gean.12164) and [Lee and Li, 2019](https://onlinelibrary.wiley.com/doi/full/10.1111/gean.12106); implemented in [SEAGAL](https://academic.oup.com/bioinformatics/article/39/7/btad431/7223197) and [SpatialDM](https://github.com/StatBiomed/SpatialDM).



### Spatial Connectivity

The way that spatially-informed methods usually work is by making use of weights based on the proximity (or spatial connectivity) between spots/cells.
These spatial connectivities are then used to calculate the metric of interest, e.g. Cosine similarity, in a spatially-informed manner.

The spatial weights in LIANA+ are by default defined as a family of radial kernels that apply a kernel to the Euclidean distance between cells/spots to bound the weights between 0 and 1, with spots that are closest having the highest spatial connectivity to one another (1), while those that are thought to be too far to be in contact are assigned 0.

Key parameters of spatial_neighbors include:
- `bandwidth` controls the radius of the spatial connectivities where higher values will result in a broader area being considered (controls the radius relative to the coordinates stored in `adata.obsm['spatial']`)
- `cutoff` controls the minimum value that will be considered to have a spatial relationship (anything lower than the `cutoff` is set to 0).
- `kernel` controls the distribution (shape) of the weights ('gaussian' by default)
- `set_diag` sets the diagonal (i.e. the weight for each spot to itself) to 1 if True. **NOTE**: Here we set it to True, as a Visium spot holds several cells, so its own expression counts (weight 1)

As choosing an optimal bandwidth can be tricky, we provide the ``query_bandwidth`` function which uses a set of coordinates to provide an estimate of how many cell or spot neighbors are being considered for each spot over a range of bandwidths. 

In [ ]:
plot, _ = li.pp.query_bandwidth(coordinates=adata.obsm['spatial'], start=0, end=300, interval_n=50)
plot

Here, we can see that a radius of 130-200 (pixels) includes 6 neighbours, i.e. the first ring of the hexagonal Visium grid (spot centres are ~119 pixels apart on this slide).

Note that `query_bandwidth` counts spots within a hard radius, while `spatial_neighbors` uses the bandwidth as the width of a Gaussian kernel and only then drops weights below `cutoff`. With `bandwidth=150` and `cutoff=0.1`, weights stay above zero up to ~2.15 × 150 ≈ 320 pixels, so each spot is connected to ~30 spots (itself included) across three rings, with weights of ~0.73, 0.3-0.4 and ~0.1. We will use a bandwidth of 150; the connectivity plot below shows this decay.

In [ ]:
li.pp.spatial_neighbors(adata, bandwidth=150, cutoff=0.1, kernel='gaussian', set_diag=True)

Let's visualize the spatial weights for a single spot to all other spots in the dataset:

In [ ]:
# weights from spot 0 to all other spots
adata.obs['connectivity'] = adata.obsp['spatial_connectivities'][:, 0].toarray().ravel()
sq.pl.spatial_scatter(adata, color='connectivity', size=1.4, cmap='viridis')
plt.show()

### Bivariate Ligand-Receptor Relationships

Now that we have covered the basics, let's see how these scores look for potential ligand-receptor interactions on our 10X Visium Slide.
Note that LIANA+ will take the presence of heteromeric complexes into account at the individual spot-level!

In [ ]:
lrdata = li.mt.bivariate(adata,
                resource_name='consensus', # uses HUMAN gene symbols
                local_name='cosine', # Name of the local function
                global_name="morans", # Name global function (or 'lee')
                n_perms=None, # None = no p-values (see below)
                mask_negatives=False, # Whether to mask LowLow/NegativeNegative interactions
                add_categories=True, # Whether to add local categories to the results
                nz_prop=0.05, # Minimum expr. proportion for ligands/receptors and their subunits
                use_raw=False,
                verbose=True
                )

In [ ]:
lrdata

#### Global scores

In [ ]:
lrdata.var.sort_values("mean", ascending=False).head(5)

The columns 'mean' and 'std' summarise the local cosine scores across all spots.

The column 'morans' represents the global Moran's R score for each ligand-receptor pair. Bivariate Moran's R values near zero imply spatial independence, while positive or negative values reflect spatial co-clustering or spatial cross-dispersion, respectively.

In [ ]:
lrdata.var.sort_values("morans", ascending=False).head()

From these Global summaries, we see that the average Cosine similarity largely represents **coverage** - e.g. *TIMP1 & CD63* is ubiquitously and uniformly distributed across the slide. 

On the other hand, **APOE^LRP1** has the highest global Moran's R. This interaction shows distinct spatial clustering, which makes it a more interesting candidate to follow up.

So, let's visualize both:

In [ ]:
sc.set_figure_params(dpi=80, dpi_save=300, format='png', frameon=False, transparent=True, figsize=[5,5])

In [ ]:
sq.pl.spatial_scatter(lrdata, color=['APOE^LRP1', 'TIMP1^CD63'], size=1.4, vmin=0, vmax=1, cmap='magma')
plt.show()

As expected, we see that the **TIMP1 & CD63** interaction is uniformly distributed across the slide, while **APOE^LRP1** shows a clear spatial pattern.

We can also see that this is the case when we look at the individual genes:

In [ ]:
sq.pl.spatial_scatter(adata, color=['APOE', 'LRP1',  'TIMP1', 'CD63'],
                       size=1.4, ncols=2)
#sq.pl.spatial_scatter(adata, color=['VTN', 'ITGAV', 'ITGB5', 'TIMP1', 'CD63'], size=1.2, ncols=2)  # you can also use `squidpy.pl.spatial_scatter` instead.
plt.show()

<div style="border-left:5px solid #f9a825; background:#fffde7; padding:10px 14px; border-radius:4px; margin:12px 0;">

**⚠️ Keep in mind:** a high local or Moran's score means that ligand and receptor are *co-localised* - expressed in the same or neighbouring 55 µm spots, each containing several cells, with spot centres ~100 µm apart. This is not proof that the cells communicate. Also, expression is spatially smooth, which on its own inflates spatial correlations.
</div>

#### Local categories
Because we set `add_categories=True`, LIANA+ also labels each spot by whether the spatially-weighted (z-scored) ligand and receptor are both high (`1`, High-High), one high and one low (`-1`), or both low (`0`, Low-Low). These are stored in `lrdata.layers['cats']`, and show that for `APOE^LRP1` not every spot with a high cosine score is High-High:

In [ ]:
sq.pl.spatial_scatter(lrdata, layer='cats', color='APOE^LRP1', size=1.4, cmap='coolwarm')
plt.show()

### Significance testing
`n_perms` controls the p-values: `None` (used above) skips them, a positive number runs that many permutations (shuffling the spots), and `0` uses an analytical approximation that is currently available only for Moran's R. So, to get global p-values cheaply, we can compute only the global scores (`local_name=None`), which returns a DataFrame:

In [ ]:
global_res = li.mt.bivariate(adata,
                             resource_name='consensus',
                             local_name=None, # global scores only
                             global_name='morans',
                             n_perms=0, # analytical p-values
                             nz_prop=0.05,
                             use_raw=False)
print((global_res['morans_pvals'] < 0.05).sum(), 'of', len(global_res), 'pairs with p < 0.05')
global_res.set_index('interaction').loc[['APOE^LRP1', 'TIMP1^CD63'], ['morans', 'morans_pvals']]

Most pairs (~70 %) are "significant", including the uniformly expressed `TIMP1^CD63`. With thousands of spatially autocorrelated spots, even tiny Moran's R values pass, and the analytical approximation is anti-conservative for small p-values. So, rank by the Moran's R itself, and use permutations (`n_perms > 0`) for p-values you report.

---

<div style="border-left:5px solid #2e7d32; background:#f1f8e9; padding:10px 14px; border-radius:4px; margin:12px 0;">

**🧪 Exercise — Does the local metric matter?**

The local scores above use `local_name='cosine'`. Re-run `li.mt.bivariate` with
`local_name='pearson'` (keep everything else the same) and compare:

1. the top 10 interactions by `mean` local score for each metric;
2. the `morans` column of both results;
3. the local score maps of `APOE^LRP1` with each metric.

Why do the rankings by `mean` differ so much, while `morans` does not change at all?
</div>

In [ ]:
# your turn
lrdata_pearson = li.mt.bivariate(adata,
                                 resource_name='consensus',
                                 local_name='pearson',
                                 global_name='morans',
                                 nz_prop=0.05,
                                 use_raw=False)

<details>
<summary><b>Solution</b></summary>

```python
top_cosine = lrdata.var.sort_values('mean', ascending=False).head(10).index
top_pearson = lrdata_pearson.var.sort_values('mean', ascending=False).head(10).index
print(pd.DataFrame({'cosine': top_cosine, 'pearson': top_pearson}))
print('shared:', len(set(top_cosine) & set(top_pearson)))

# Moran's R is computed from the expression and the spatial weights only
print((lrdata.var['morans'] - lrdata_pearson.var['morans']).abs().max())

sq.pl.spatial_scatter(lrdata, color='APOE^LRP1', size=1.4, cmap='magma', title='cosine')
sq.pl.spatial_scatter(lrdata_pearson, color='APOE^LRP1', size=1.4, cmap='coolwarm',
                      vmin=-1, vmax=1, title='pearson')
plt.show()
```

The two top-10 lists share **no** interactions. Weighted cosine similarity works on
non-negative expression, so a high mean score mostly means *both genes are expressed
everywhere* (coverage, e.g. `TIMP1^CD63`). Weighted Pearson centres each gene on its
neighbourhood mean, so it ranges from −1 to 1 and asks whether ligand and receptor rise and
fall together **across neighbouring spots** — its top list is dominated by collagen–integrin
pairs (`COL1A1^ITGA11_ITGB1`, …). That is also why its map needs a diverging colour
scale centred on 0, while cosine uses a sequential one.

`morans` is identical because the global score is computed independently of the local
function. So the local metric is a modelling choice about *what "co-expressed here" means*;
pick it for the question you are asking, and use the global score to find spatially
variable pairs.

</details>

## Beyond Ligand-Receptors (optional)

While protein-mediated ligand-receptor interactions are interesting, cell-cell communication is not limited to those alone. Rather it is a complex process that involves a variety of different mechanisms such as signalling pathways, metabolite-mediated signalling, and distinct cell types.

So, if such diverse mechanisms are involved in cell-cell communication, why should we limit ourselves to ligand-receptor interactions?
Let's see how we can use LIANA+ to explore other types of cell-cell communication.

One simple approach would be to check relationships e.g. between transcription factors and cell type proportions.

### Extract Cell type Composition
This slide comes with estimated cell type proportions using cell2location; See [Kuppe et al., 2022](https://www.nature.com/articles/s41586-022-05060-x). Let's extract from .obsm them to an independent AnnData object.

In [ ]:
# let's extract those
comps = li.pp.obsm_to_adata(adata, 'compositions')
# check key cell types
sq.pl.spatial_scatter(comps, color=['vSMCs','CM', 'Endo', 'Fib'], size=1.3, ncols=2)
plt.show()

### Estimate Transcription Factor Activity

In [ ]:
# Get transcription factor resource
net = dc.op.collectri(organism='human', remove_complexes=True, license='academic', verbose=False)

While multi-omics datasets might be even more of an interest, for the sake of simplicity (and because the general lack of spatial multi-omics data at current times), let's instead use enrichment analysis to estimate the activity of transcription factors in each spot. We will use one of [decoupler-py's](https://decoupler-py.readthedocs.io/en/latest/index.html) enrichment methods with [CollectTRI](https://www.biorxiv.org/content/10.1101/2023.03.30.534849v1.abstract) to do so. Refer to this [tutorial](https://decoupler-py.readthedocs.io/en/latest/notebooks/dorothea.html) for more info.

In [ ]:
# Estimate activities
dc.mt.ulm(
    data=adata,
    net=net,
    bsize=128,  
    tmin = 50,
    verbose=True,
    raw=False
)

#### Extract highly-variable TF activities
To reduce the number of TFs for the sake of computational speed, we will only focus on the top 50 most variable TFs.

ULM scores are not strictly positive and some TFs have a mean close to 0, so a coefficient of variation (std / mean) would favour those. Instead, we simply select the TFs with the highest standard deviation across spots.


In [ ]:
est = li.pp.obsm_to_adata(adata, 'score_ulm')
est.var['std'] = est.X.std(axis=0)
top_tfs = est.var.sort_values('std', ascending=False).head(50).index


Create MuData object with TF activities and cell type proportions, and transfer spatial connectivities and other information from the original AnnData object.

In [ ]:
import mudata as mu
mdata = mu.MuData({"tf":est, "comps":comps})
mdata.obsp = adata.obsp
mdata.uns = adata.uns
mdata.obsm = adata.obsm

Define Interactions of interest:

In [ ]:
from itertools import product

In [ ]:
interactions = list(product(comps.var.index, top_tfs))

In [ ]:
interactions[:5]

### Estimate Cosine Similarity

In [ ]:
bdata = li.mt.bivariate(mdata,
                        x_mod="comps",
                        y_mod="tf",
                        x_transform=sc.pp.scale,
                        y_transform=sc.pp.scale,
                        local_name="cosine", 
                        interactions=interactions,
                        mask_negatives=True, 
                        add_categories=True,
                        x_use_raw=False,
                        y_use_raw=False,
                        xy_sep="<->",
                        x_name='celltype',
                        y_name='tf'
                        )

<div class="alert alert-info">

To make the distributions comparable, we simply z-scale the TF activities and cell type proportions via the `x_transform` & `y_transform` parameters.

The type of transformation will affect the interpretation of the results, and different types of transformation might be more appropriate for different types of data. We provide zero-inflated minmax `zi_minmax` & `neg_to_zero` transformation functions via `li.fun.transform`.


</div>

In [ ]:
bdata.var.sort_values("mean", ascending=False).head(5)

#### Let's plot the results

In [ ]:
sq.pl.spatial_scatter(bdata, color=['CM<->MEF2C', 'Fib<->CTNNB1'], size=1.4, cmap="coolwarm", vmax=1, vmin=-1)
plt.show()

Cosine similarity on z-scored data is also high where *both* variables are low (Low-Low), which is why we set `mask_negatives=True`: grey spots are those that are not High-High. That cardiomyocyte proportions top the list together with the cardiac TFs MEF2C and NKX2-5 is a sanity check rather than a discovery.

In [ ]:
sq.pl.spatial_scatter(mdata.mod['tf'], color=['MEF2C', 'CTNNB1'], cmap='coolwarm', size=1.4, vcenter=0)
plt.show()

In [ ]:
sq.pl.spatial_scatter(mdata.mod['comps'], color=['CM', 'Fib'], cmap='viridis', size=1.4)
plt.show()

---

**Next:** `02_misty.ipynb` moves beyond pairs of variables and learns spatial relationships with MISTy.